# Cross-Analyst Join Tool & Cortex Agent
Joins results across multiple Cortex Analyst semantic views using predefined join keys, with a Cortex Agent for smart auto-routing.

*Co-authored with CoCo*

In [ ]:
%%sql -r create_db
-- STEP 1: Create database and sample tables
CREATE DATABASE IF NOT EXISTS CORTEX_ANALYST_DEMO;

In [ ]:
%%sql -r products_table
CREATE OR REPLACE TABLE CORTEX_ANALYST_DEMO.PUBLIC.PRODUCTS (
    PRODUCT_ID INT,
    PRODUCT_NAME VARCHAR(200),
    CATEGORY VARCHAR(100),
    UNIT_PRICE DECIMAL(10,2),
    SUPPLIER_ID INT,
    CREATED_DATE DATE
);

INSERT INTO CORTEX_ANALYST_DEMO.PUBLIC.PRODUCTS VALUES
(1, 'Wireless Mouse', 'Electronics', 29.99, 101, '2024-01-15'),
(2, 'Mechanical Keyboard', 'Electronics', 89.99, 101, '2024-01-20'),
(3, 'USB-C Hub', 'Electronics', 49.99, 102, '2024-02-01'),
(4, 'Standing Desk', 'Furniture', 399.99, 103, '2024-02-10'),
(5, 'Ergonomic Chair', 'Furniture', 599.99, 103, '2024-02-15'),
(6, 'Monitor Arm', 'Accessories', 79.99, 102, '2024-03-01'),
(7, 'Webcam HD', 'Electronics', 69.99, 104, '2024-03-10'),
(8, 'Noise-Cancel Headphones', 'Electronics', 199.99, 104, '2024-03-15'),
(9, 'Desk Lamp', 'Accessories', 45.99, 105, '2024-04-01'),
(10, 'Cable Management Kit', 'Accessories', 19.99, 105, '2024-04-05');

In [ ]:
%%sql -r inventory_table
CREATE OR REPLACE TABLE CORTEX_ANALYST_DEMO.PUBLIC.INVENTORY (
    INVENTORY_ID INT,
    PRODUCT_ID INT,
    WAREHOUSE_LOCATION VARCHAR(100),
    QUANTITY_ON_HAND INT,
    REORDER_LEVEL INT,
    LAST_RESTOCK_DATE DATE
);

INSERT INTO CORTEX_ANALYST_DEMO.PUBLIC.INVENTORY VALUES
(1, 1, 'Warehouse-A', 250, 50, '2024-06-01'),
(2, 2, 'Warehouse-A', 120, 30, '2024-06-05'),
(3, 3, 'Warehouse-B', 300, 75, '2024-06-10'),
(4, 4, 'Warehouse-C', 45, 10, '2024-06-15'),
(5, 5, 'Warehouse-C', 30, 8, '2024-06-20'),
(6, 6, 'Warehouse-B', 200, 40, '2024-07-01'),
(7, 7, 'Warehouse-A', 180, 50, '2024-07-05'),
(8, 8, 'Warehouse-A', 90, 25, '2024-07-10'),
(9, 9, 'Warehouse-B', 400, 100, '2024-07-15'),
(10, 10, 'Warehouse-B', 500, 150, '2024-07-20');

In [ ]:
%%sql -r suppliers_table
CREATE OR REPLACE TABLE CORTEX_ANALYST_DEMO.PUBLIC.SUPPLIERS (
    SUPPLIER_ID INT,
    SUPPLIER_NAME VARCHAR(200),
    CONTACT_EMAIL VARCHAR(200),
    COUNTRY VARCHAR(100),
    LEAD_TIME_DAYS INT,
    RATING DECIMAL(3,2)
);

INSERT INTO CORTEX_ANALYST_DEMO.PUBLIC.SUPPLIERS VALUES
(101, 'TechParts Inc', 'sales@techparts.com', 'USA', 5, 4.5),
(102, 'GlobalConnect Ltd', 'orders@globalconnect.com', 'Germany', 7, 4.2),
(103, 'FurniturePro', 'supply@furniturepro.com', 'Canada', 14, 4.8),
(104, 'AudioVision Corp', 'wholesale@audiovision.com', 'Japan', 10, 4.6),
(105, 'OfficeEssentials', 'bulk@officeessentials.com', 'USA', 3, 4.0);

## Step 2: Create Semantic Views
Each table gets its own semantic view with dimensions and measures defined.

In [ ]:
%%sql -r sv_products
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'CORTEX_ANALYST_DEMO.PUBLIC',
  $$
name: SV_PRODUCTS
description: "Semantic view for the products catalog including product details, categories, and pricing"
tables:
  - name: PRODUCTS
    base_table:
      database: CORTEX_ANALYST_DEMO
      schema: PUBLIC
      table: PRODUCTS
    primary_key:
      columns:
        - PRODUCT_ID
    dimensions:
      - name: PRODUCT_ID
        description: "Unique product identifier"
        expr: PRODUCT_ID
        data_type: NUMBER
      - name: PRODUCT_NAME
        description: "Name of the product"
        expr: PRODUCT_NAME
        data_type: TEXT
      - name: CATEGORY
        description: "Product category (Electronics, Furniture, Accessories)"
        expr: CATEGORY
        data_type: TEXT
      - name: SUPPLIER_ID
        description: "Foreign key to the suppliers table"
        expr: SUPPLIER_ID
        data_type: NUMBER
      - name: CREATED_DATE
        description: "Date the product was added to catalog"
        expr: CREATED_DATE
        data_type: DATE
    measures:
      - name: UNIT_PRICE
        description: "Price per unit in USD"
        expr: UNIT_PRICE
        data_type: NUMBER
      - name: PRODUCT_COUNT
        description: "Count of products"
        expr: COUNT(PRODUCT_ID)
        data_type: NUMBER
      - name: AVG_PRICE
        description: "Average product price"
        expr: AVG(UNIT_PRICE)
        data_type: NUMBER
$$
);

In [ ]:
%%sql -r sv_inventory
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'CORTEX_ANALYST_DEMO.PUBLIC',
  $$
name: SV_INVENTORY
description: "Semantic view for inventory levels, warehouse locations, and stock management"
tables:
  - name: INVENTORY
    base_table:
      database: CORTEX_ANALYST_DEMO
      schema: PUBLIC
      table: INVENTORY
    primary_key:
      columns:
        - INVENTORY_ID
    dimensions:
      - name: INVENTORY_ID
        description: "Unique inventory record identifier"
        expr: INVENTORY_ID
        data_type: NUMBER
      - name: PRODUCT_ID
        description: "Foreign key to the products table"
        expr: PRODUCT_ID
        data_type: NUMBER
      - name: WAREHOUSE_LOCATION
        description: "Physical warehouse location (Warehouse-A, Warehouse-B, Warehouse-C)"
        expr: WAREHOUSE_LOCATION
        data_type: TEXT
      - name: LAST_RESTOCK_DATE
        description: "Date the item was last restocked"
        expr: LAST_RESTOCK_DATE
        data_type: DATE
    measures:
      - name: QUANTITY_ON_HAND
        description: "Current quantity available in stock"
        expr: QUANTITY_ON_HAND
        data_type: NUMBER
      - name: REORDER_LEVEL
        description: "Minimum quantity threshold before reorder is needed"
        expr: REORDER_LEVEL
        data_type: NUMBER
      - name: TOTAL_STOCK
        description: "Total quantity across all items"
        expr: SUM(QUANTITY_ON_HAND)
        data_type: NUMBER
      - name: ITEMS_BELOW_REORDER
        description: "Count of items below reorder level"
        expr: COUNT(CASE WHEN QUANTITY_ON_HAND <= REORDER_LEVEL THEN 1 END)
        data_type: NUMBER
$$
);

In [ ]:
%%sql -r sv_suppliers
CALL SYSTEM$CREATE_SEMANTIC_VIEW_FROM_YAML(
  'CORTEX_ANALYST_DEMO.PUBLIC',
  $$
name: SV_SUPPLIERS
description: "Semantic view for supplier information including contact details, lead times, and ratings"
tables:
  - name: SUPPLIERS
    base_table:
      database: CORTEX_ANALYST_DEMO
      schema: PUBLIC
      table: SUPPLIERS
    primary_key:
      columns:
        - SUPPLIER_ID
    dimensions:
      - name: SUPPLIER_ID
        description: "Unique supplier identifier"
        expr: SUPPLIER_ID
        data_type: NUMBER
      - name: SUPPLIER_NAME
        description: "Name of the supplier company"
        expr: SUPPLIER_NAME
        data_type: TEXT
      - name: CONTACT_EMAIL
        description: "Supplier contact email address"
        expr: CONTACT_EMAIL
        data_type: TEXT
      - name: COUNTRY
        description: "Country where the supplier is located"
        expr: COUNTRY
        data_type: TEXT
    measures:
      - name: LEAD_TIME_DAYS
        description: "Number of days for delivery from supplier"
        expr: LEAD_TIME_DAYS
        data_type: NUMBER
      - name: RATING
        description: "Supplier quality rating from 1.0 to 5.0"
        expr: RATING
        data_type: NUMBER
      - name: AVG_LEAD_TIME
        description: "Average lead time across suppliers"
        expr: AVG(LEAD_TIME_DAYS)
        data_type: NUMBER
      - name: AVG_RATING
        description: "Average supplier rating"
        expr: AVG(RATING)
        data_type: NUMBER
$$
);

## Step 3: Create Join Config Table
This table defines how semantic views relate to each other via join keys.

In [ ]:
%%sql -r join_config
CREATE OR REPLACE TABLE CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG (
    CONFIG_ID INT AUTOINCREMENT,
    LEFT_SEMANTIC_VIEW VARCHAR(500) COMMENT 'Fully qualified name of the left semantic view (DB.SCHEMA.VIEW)',
    LEFT_TABLE VARCHAR(200) COMMENT 'Table name within the left semantic view',
    LEFT_JOIN_KEY VARCHAR(200) COMMENT 'Column name in the left table to join on',
    RIGHT_SEMANTIC_VIEW VARCHAR(500) COMMENT 'Fully qualified name of the right semantic view (DB.SCHEMA.VIEW)',
    RIGHT_TABLE VARCHAR(200) COMMENT 'Table name within the right semantic view',
    RIGHT_JOIN_KEY VARCHAR(200) COMMENT 'Column name in the right table to join on',
    JOIN_TYPE VARCHAR(20) DEFAULT 'INNER' COMMENT 'Type of join: INNER, LEFT, RIGHT, FULL',
    RELATIONSHIP_NAME VARCHAR(200) COMMENT 'Human-readable name for this relationship',
    IS_ACTIVE BOOLEAN DEFAULT TRUE COMMENT 'Whether this join config is active',
    CREATED_AT TIMESTAMP DEFAULT CURRENT_TIMESTAMP()
);

INSERT INTO CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG 
(LEFT_SEMANTIC_VIEW, LEFT_TABLE, LEFT_JOIN_KEY, RIGHT_SEMANTIC_VIEW, RIGHT_TABLE, RIGHT_JOIN_KEY, JOIN_TYPE, RELATIONSHIP_NAME)
VALUES
-- Products <-> Inventory (via PRODUCT_ID)
('CORTEX_ANALYST_DEMO.PUBLIC.SV_PRODUCTS', 'PRODUCTS', 'PRODUCT_ID', 
 'CORTEX_ANALYST_DEMO.PUBLIC.SV_INVENTORY', 'INVENTORY', 'PRODUCT_ID', 
 'INNER', 'Product Inventory Levels'),
-- Products <-> Suppliers (via SUPPLIER_ID)
('CORTEX_ANALYST_DEMO.PUBLIC.SV_PRODUCTS', 'PRODUCTS', 'SUPPLIER_ID', 
 'CORTEX_ANALYST_DEMO.PUBLIC.SV_SUPPLIERS', 'SUPPLIERS', 'SUPPLIER_ID', 
 'INNER', 'Product Supplier Details');

In [ ]:
%%sql -r config_view
SELECT * FROM CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG;

## Step 4: Create Cross-Analyst Join Procedure
This stored procedure:
1. Reads each semantic view's YAML definition
2. Uses Cortex COMPLETE (LLM) to generate SQL for each view's table
3. Executes each generated query into a temp table
4. Joins temp tables using keys from the config table
5. Stores the final result in a destination table

In [ ]:
%%sql -r create_proc
CREATE OR REPLACE PROCEDURE CORTEX_ANALYST_DEMO.PUBLIC.CROSS_ANALYST_JOIN(
    P_QUESTION VARCHAR,
    P_SEMANTIC_VIEWS ARRAY,
    P_RESULT_TABLE VARCHAR DEFAULT 'CORTEX_ANALYST_DEMO.PUBLIC.CROSS_ANALYST_RESULTS',
    P_MODEL VARCHAR DEFAULT 'llama3.1-405b'
)
RETURNS VARCHAR
LANGUAGE SQL
EXECUTE AS CALLER
AS
$$
DECLARE
    v_view_count INT;
    v_current_view VARCHAR;
    v_generated_sql VARCHAR;
    v_temp_table_name VARCHAR;
    v_idx INT DEFAULT 0;
    v_final_sql VARCHAR;
    v_msg VARCHAR DEFAULT '';
    v_join_count INT DEFAULT 0;
    v_sv_yaml VARCHAR;
    v_prompt VARCHAR;
    v_left_idx INT;
    v_right_idx INT;
    v_select_cols VARCHAR DEFAULT '';
BEGIN
    v_view_count := ARRAY_SIZE(:P_SEMANTIC_VIEWS);
    IF (v_view_count < 2) THEN
        RETURN 'Error: At least 2 semantic views are required.';
    END IF;

    -- Step 1: Generate and execute SQL for each semantic view
    FOR v_idx IN 0 TO v_view_count - 1 DO
        v_current_view := :P_SEMANTIC_VIEWS[v_idx]::VARCHAR;
        v_temp_table_name := 'CORTEX_ANALYST_DEMO.PUBLIC.TEMP_ANALYST_' || v_idx::VARCHAR;

        LET yaml_rs RESULTSET := (
            SELECT SYSTEM$READ_YAML_FROM_SEMANTIC_VIEW(:v_current_view) AS sv_yaml
        );
        LET yaml_cur CURSOR FOR yaml_rs;
        OPEN yaml_cur;
        FETCH yaml_cur INTO v_sv_yaml;
        CLOSE yaml_cur;

        v_prompt := 'You are a Snowflake SQL expert. Generate a SELECT query for ONLY the single table defined in this semantic view YAML. '
            || 'CRITICAL RULES: '
            || '1) Query ONLY the ONE table defined in base_table. Do NOT join to other tables. '
            || '2) Return ONLY the raw SQL - no markdown, no backticks, no explanation, no comments, no semicolons. '
            || '3) SELECT ALL columns defined as dimensions and measures (use the expr values). Include primary key columns. '
            || '4) Use exact column names from the YAML expr fields. '
            || '5) Do NOT use aggregate functions unless the question specifically asks for totals/averages/counts. '
            || '6) For simple listing queries, select all dimension columns and raw measure columns without aggregation. '
            || CHR(10) || CHR(10) || 'Semantic View YAML:' || CHR(10) || v_sv_yaml
            || CHR(10) || CHR(10) || 'User Question: ' || :P_QUESTION
            || CHR(10) || CHR(10) || 'Return ONLY the SQL query for this single table:';

        LET llm_rs RESULTSET := (
            SELECT SNOWFLAKE.CORTEX.COMPLETE(:P_MODEL, :v_prompt) AS llm_sql
        );
        LET llm_cur CURSOR FOR llm_rs;
        OPEN llm_cur;
        FETCH llm_cur INTO v_generated_sql;
        CLOSE llm_cur;

        IF (v_generated_sql IS NULL OR LENGTH(TRIM(v_generated_sql)) < 5) THEN
            v_msg := v_msg || 'ERROR: No SQL for ' || v_current_view || '. ';
            CONTINUE;
        END IF;

        v_generated_sql := TRIM(v_generated_sql);
        IF (STARTSWITH(v_generated_sql, '```')) THEN
            v_generated_sql := REGEXP_REPLACE(v_generated_sql, '^```(sql)?[\\s]*', '');
            v_generated_sql := REGEXP_REPLACE(v_generated_sql, '[\\s]*```$', '');
            v_generated_sql := TRIM(v_generated_sql);
        END IF;
        IF (ENDSWITH(v_generated_sql, ';')) THEN
            v_generated_sql := LEFT(v_generated_sql, LENGTH(v_generated_sql) - 1);
        END IF;

        EXECUTE IMMEDIATE 'CREATE OR REPLACE TEMPORARY TABLE ' || v_temp_table_name || ' AS ' || v_generated_sql;
        v_msg := v_msg || 'SV[' || v_idx::VARCHAR || '] OK; ';
    END FOR;

    -- Step 2: Build column list (avoid duplicate join key columns)
    LET cols0_rs RESULTSET := (
        SELECT COLUMN_NAME FROM CORTEX_ANALYST_DEMO.INFORMATION_SCHEMA.COLUMNS
        WHERE TABLE_SCHEMA = 'PUBLIC' AND TABLE_NAME = 'TEMP_ANALYST_0'
        ORDER BY ORDINAL_POSITION
    );
    FOR col_rec IN cols0_rs DO
        IF (LENGTH(v_select_cols) > 0) THEN
            v_select_cols := v_select_cols || ', ';
        END IF;
        v_select_cols := v_select_cols || 't0.' || col_rec.COLUMN_NAME;
    END FOR;

    FOR v_idx IN 1 TO v_view_count - 1 DO
        LET cols_rs RESULTSET := (EXECUTE IMMEDIATE
            'SELECT COLUMN_NAME FROM CORTEX_ANALYST_DEMO.INFORMATION_SCHEMA.COLUMNS '
            || 'WHERE TABLE_SCHEMA = ''PUBLIC'' AND TABLE_NAME = ''TEMP_ANALYST_' || v_idx::VARCHAR || ''' '
            || 'AND COLUMN_NAME NOT IN ('
            || '  SELECT LEFT_JOIN_KEY FROM CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG WHERE IS_ACTIVE = TRUE'
            || '  UNION SELECT RIGHT_JOIN_KEY FROM CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG WHERE IS_ACTIVE = TRUE'
            || ') ORDER BY ORDINAL_POSITION'
        );
        FOR col_rec IN cols_rs DO
            v_select_cols := v_select_cols || ', t' || v_idx::VARCHAR || '.' || col_rec.COLUMN_NAME;
        END FOR;
    END FOR;

    -- Step 3: Build FROM + JOIN clauses
    v_final_sql := 'SELECT ' || v_select_cols || ' FROM CORTEX_ANALYST_DEMO.PUBLIC.TEMP_ANALYST_0 AS t0';

    LET join_rs RESULTSET := (
        SELECT JOIN_TYPE, LEFT_JOIN_KEY, RIGHT_JOIN_KEY, LEFT_SEMANTIC_VIEW, RIGHT_SEMANTIC_VIEW
        FROM CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG
        WHERE IS_ACTIVE = TRUE
    );
    FOR rec IN join_rs DO
        v_left_idx := ARRAY_POSITION(rec.LEFT_SEMANTIC_VIEW::VARIANT, :P_SEMANTIC_VIEWS);
        v_right_idx := ARRAY_POSITION(rec.RIGHT_SEMANTIC_VIEW::VARIANT, :P_SEMANTIC_VIEWS);
        
        IF (v_left_idx IS NOT NULL AND v_right_idx IS NOT NULL) THEN
            v_join_count := v_join_count + 1;
            v_final_sql := v_final_sql || ' ' || rec.JOIN_TYPE || ' JOIN '
                || 'CORTEX_ANALYST_DEMO.PUBLIC.TEMP_ANALYST_' || v_right_idx::VARCHAR || ' AS t' || v_right_idx::VARCHAR
                || ' ON t' || v_left_idx::VARCHAR || '.' || rec.LEFT_JOIN_KEY 
                || ' = t' || v_right_idx::VARCHAR || '.' || rec.RIGHT_JOIN_KEY;
        END IF;
    END FOR;

    IF (v_join_count = 0) THEN
        RETURN 'Error: No join config found for these views. ' || v_msg;
    END IF;

    -- Step 4: Execute final join and store results
    EXECUTE IMMEDIATE 'CREATE OR REPLACE TABLE ' || :P_RESULT_TABLE || ' AS ' || v_final_sql;

    RETURN 'Success! Joined ' || v_view_count::VARCHAR || ' views (' || v_join_count::VARCHAR 
        || ' joins). Results in: ' || :P_RESULT_TABLE || ' | ' || v_msg;
EXCEPTION
    WHEN OTHER THEN
        RETURN 'Error: ' || SQLERRM || ' | ' || v_msg || ' | Generated SQL: ' || COALESCE(v_generated_sql, 'N/A') || ' | Join SQL: ' || COALESCE(v_final_sql, 'N/A');
END;
$$;

## Step 5: Helper Procedure to Manage Join Configs
Add, remove, or list join relationships.

In [ ]:
%%sql -r manage_proc
CREATE OR REPLACE PROCEDURE CORTEX_ANALYST_DEMO.PUBLIC.MANAGE_JOIN_CONFIG(
    P_ACTION VARCHAR,
    P_LEFT_VIEW VARCHAR DEFAULT NULL,
    P_LEFT_TABLE VARCHAR DEFAULT NULL,
    P_LEFT_KEY VARCHAR DEFAULT NULL,
    P_RIGHT_VIEW VARCHAR DEFAULT NULL,
    P_RIGHT_TABLE VARCHAR DEFAULT NULL,
    P_RIGHT_KEY VARCHAR DEFAULT NULL,
    P_JOIN_TYPE VARCHAR DEFAULT 'INNER',
    P_RELATIONSHIP_NAME VARCHAR DEFAULT NULL
)
RETURNS TABLE(CONFIG_ID INT, LEFT_VIEW VARCHAR, LEFT_KEY VARCHAR, RIGHT_VIEW VARCHAR, RIGHT_KEY VARCHAR, JOIN_TYPE VARCHAR, RELATIONSHIP VARCHAR, ACTIVE BOOLEAN)
LANGUAGE SQL
EXECUTE AS CALLER
AS
$$
DECLARE
    v_result VARCHAR;
BEGIN
    CASE UPPER(:P_ACTION)
        WHEN 'ADD' THEN
            INSERT INTO CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG
                (LEFT_SEMANTIC_VIEW, LEFT_TABLE, LEFT_JOIN_KEY, RIGHT_SEMANTIC_VIEW, RIGHT_TABLE, RIGHT_JOIN_KEY, JOIN_TYPE, RELATIONSHIP_NAME)
            VALUES (:P_LEFT_VIEW, :P_LEFT_TABLE, :P_LEFT_KEY, :P_RIGHT_VIEW, :P_RIGHT_TABLE, :P_RIGHT_KEY, :P_JOIN_TYPE, :P_RELATIONSHIP_NAME);
        WHEN 'REMOVE' THEN
            UPDATE CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG
            SET IS_ACTIVE = FALSE
            WHERE LEFT_SEMANTIC_VIEW = :P_LEFT_VIEW AND RIGHT_SEMANTIC_VIEW = :P_RIGHT_VIEW;
        WHEN 'LIST' THEN
            NULL;
        ELSE
            NULL;
    END CASE;

    LET rs RESULTSET := (
        SELECT CONFIG_ID, LEFT_SEMANTIC_VIEW AS LEFT_VIEW, LEFT_JOIN_KEY AS LEFT_KEY, 
               RIGHT_SEMANTIC_VIEW AS RIGHT_VIEW, RIGHT_JOIN_KEY AS RIGHT_KEY, 
               JOIN_TYPE, RELATIONSHIP_NAME AS RELATIONSHIP, IS_ACTIVE AS ACTIVE
        FROM CORTEX_ANALYST_DEMO.PUBLIC.SEMANTIC_VIEW_JOIN_CONFIG
        ORDER BY CONFIG_ID
    );
    RETURN TABLE(rs);
END;
$$;

## Step 6: Create the Cortex Agent
The CROSS_DOMAIN_AGENT auto-routes questions:
- **Single-domain** → routes to the specific analyst tool
- **Cross-domain** → queries multiple analyst tools and combines results

In [ ]:
%%sql -r grant_agent
USE ROLE ACCOUNTADMIN;
GRANT CREATE AGENT ON SCHEMA CORTEX_ANALYST_DEMO.PUBLIC TO ROLE SNOWFLAKE_INTELLIGENCE_ADMIN;

In [ ]:
%%sql -r create_agent
CREATE OR REPLACE AGENT CORTEX_ANALYST_DEMO.PUBLIC.CROSS_DOMAIN_AGENT
  FROM SPECIFICATION $$
instructions:
  system: |
    You are a cross-domain supply chain analyst agent. You help users query product, inventory, and supplier data.
  orchestration: |
    ROUTING RULES:

    1. Single-domain questions - Route to the specific analyst tool:
       - Product details, pricing, categories → use "products_analyst"
       - Stock levels, warehouse locations, reorder → use "inventory_analyst"
       - Supplier info, lead times, ratings, contacts → use "suppliers_analyst"

    2. Cross-domain questions - If the question spans MULTIPLE domains (e.g., "which products
       from US suppliers have low stock"), call EACH relevant analyst tool to get partial data.
       The join keys are:
       - PRODUCT_ID links Products and Inventory
       - SUPPLIER_ID links Products and Suppliers
  response: |
    Always show results in a clear table format.
    For cross-domain queries, explain which data sources you combined.

tools:
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: products_analyst
      description: "Use for questions about product catalog: product names, categories (Electronics, Furniture, Accessories), unit prices, and when products were added. Has PRODUCT_ID and SUPPLIER_ID."
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: inventory_analyst
      description: "Use for questions about inventory and stock: quantity on hand, warehouse locations (Warehouse-A/B/C), reorder levels, and restock dates. Has PRODUCT_ID for joining with products."
  - tool_spec:
      type: cortex_analyst_text_to_sql
      name: suppliers_analyst
      description: "Use for questions about suppliers: supplier names, contact emails, countries, delivery lead times, and quality ratings. Has SUPPLIER_ID for joining with products."

tool_resources:
  products_analyst:
    semantic_view: CORTEX_ANALYST_DEMO.PUBLIC.SV_PRODUCTS
  inventory_analyst:
    semantic_view: CORTEX_ANALYST_DEMO.PUBLIC.SV_INVENTORY
  suppliers_analyst:
    semantic_view: CORTEX_ANALYST_DEMO.PUBLIC.SV_SUPPLIERS
$$;

## Step 7: Test the Agent
Test with single-domain and cross-domain questions.

In [ ]:
%%sql -r test_single
-- Single-domain: routes to products_analyst only
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'CORTEX_ANALYST_DEMO.PUBLIC.CROSS_DOMAIN_AGENT',
    '{"messages": [{"role": "user", "content": [{"type": "text", "text": "What are the top 3 most expensive products?"}]}]}'
) AS response;

In [ ]:
%%sql -r test_cross
-- Cross-domain: routes to products + suppliers analysts
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'CORTEX_ANALYST_DEMO.PUBLIC.CROSS_DOMAIN_AGENT',
    '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Show me all products from US suppliers with their prices and lead times"}]}]}'
) AS response;

In [ ]:
%%sql -r test_3way
-- 3-way cross-domain: all three analysts
SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(
    'CORTEX_ANALYST_DEMO.PUBLIC.CROSS_DOMAIN_AGENT',
    '{"messages": [{"role": "user", "content": [{"type": "text", "text": "Which electronics products from Japanese suppliers have stock below 100 units?"}]}]}'
) AS response;

## Step 8: Programmatic Cross-Analyst Join (Alternative)
For batch/pipeline use cases where you want explicit SQL-level joins between analyst-generated queries stored in a result table.

In [ ]:
%%sql -r join_2way
-- 2-way join: Products + Inventory
CALL CORTEX_ANALYST_DEMO.PUBLIC.CROSS_ANALYST_JOIN(
    'Show me all electronics products with their stock quantities',
    ARRAY_CONSTRUCT('CORTEX_ANALYST_DEMO.PUBLIC.SV_PRODUCTS', 
                    'CORTEX_ANALYST_DEMO.PUBLIC.SV_INVENTORY')
);

In [ ]:
%%sql -r results_2way
SELECT * FROM CORTEX_ANALYST_DEMO.PUBLIC.CROSS_ANALYST_RESULTS;

In [ ]:
%%sql -r join_3way
-- 3-way join: Products + Inventory + Suppliers
CALL CORTEX_ANALYST_DEMO.PUBLIC.CROSS_ANALYST_JOIN(
    'Show me all products with their supplier names and stock levels',
    ARRAY_CONSTRUCT('CORTEX_ANALYST_DEMO.PUBLIC.SV_PRODUCTS', 
                    'CORTEX_ANALYST_DEMO.PUBLIC.SV_INVENTORY',
                    'CORTEX_ANALYST_DEMO.PUBLIC.SV_SUPPLIERS'),
    'CORTEX_ANALYST_DEMO.PUBLIC.FULL_SUPPLY_CHAIN_VIEW'
);

In [ ]:
%%sql -r results_3way
SELECT * FROM CORTEX_ANALYST_DEMO.PUBLIC.FULL_SUPPLY_CHAIN_VIEW;

## Step 9: Manage Join Configurations
Use the helper procedure to add/remove/list join relationships.

In [ ]:
%%sql -r list_configs
-- List all configured join relationships
CALL CORTEX_ANALYST_DEMO.PUBLIC.MANAGE_JOIN_CONFIG('LIST');

In [ ]:
%%sql -r add_example
-- Example: Add a new join relationship (uncomment to run)
-- CALL CORTEX_ANALYST_DEMO.PUBLIC.MANAGE_JOIN_CONFIG(
--     'ADD',
--     'DB.SCHEMA.SV_ORDERS', 'ORDERS', 'CUSTOMER_ID',
--     'DB.SCHEMA.SV_CUSTOMERS', 'CUSTOMERS', 'CUSTOMER_ID',
--     'LEFT', 'Order Customer Details'
-- );
SELECT 'Ready to add new relationships' AS status;